In [93]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow

In [94]:
weekly_hvac_permits = pd.read_parquet("../data/weekly_hvac_permits.parquet")

In [95]:
weekly_hvac_permits.index.dtype

dtype('int64')

In [96]:
weekly_hvac_permits

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
867,2026-08-23,385,la_hvac
868,2026-08-30,372,la_hvac
869,2026-09-06,358,la_hvac
870,2026-09-13,299,la_hvac


In [97]:
mask = weekly_hvac_permits['ds'] < '2026-03-22'
print(mask)

0       True
1       True
2       True
3       True
4       True
       ...  
867    False
868    False
869    False
870    False
871    False
Name: ds, Length: 872, dtype: bool


In [98]:
weekly_hvac_permits.loc[mask]

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
840,2026-02-15,281,la_hvac
841,2026-02-22,281,la_hvac
842,2026-03-01,302,la_hvac
843,2026-03-08,282,la_hvac


In [99]:
used_set = weekly_hvac_permits.iloc[:-26]

In [100]:
used_set

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
841,2026-02-22,281,la_hvac
842,2026-03-01,302,la_hvac
843,2026-03-08,282,la_hvac
844,2026-03-15,270,la_hvac


In [101]:
origins = used_set['ds'].iloc[793:838:4]
print(origins)

793   2025-03-23
797   2025-04-20
801   2025-05-18
805   2025-06-15
809   2025-07-13
813   2025-08-10
817   2025-09-07
821   2025-10-05
825   2025-11-02
829   2025-11-30
833   2025-12-28
837   2026-01-25
Name: ds, dtype: datetime64[us]


In [102]:
mask = weekly_hvac_permits['ds'] <= origins.iloc[0]

In [103]:
train = weekly_hvac_permits.loc[mask]

In [104]:
train

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
789,2025-02-23,198,la_hvac
790,2025-03-02,259,la_hvac
791,2025-03-09,247,la_hvac
792,2025-03-16,258,la_hvac


In [105]:
akey = used_set.iloc[origins.index[0] + 1: origins.index[0] + 7]

In [106]:
akey

,ds,y,unique_id
794,2025-03-30,233,la_hvac
795,2025-04-06,238,la_hvac
796,2025-04-13,306,la_hvac
797,2025-04-20,242,la_hvac
798,2025-04-27,268,la_hvac
799,2025-05-04,249,la_hvac


In [107]:
akey_guesses = used_set['y'].iloc[origins.index[0] - (52 - 1) : origins.index[0] - (52 - 7)]

In [108]:
akey_guesses

742    229
743    308
744    328
745    301
746    305
747    245
Name: y, dtype: int64

In [109]:
akey['baseline'] = akey_guesses.tolist()

In [110]:
akey

,ds,y,unique_id,baseline
794,2025-03-30,233,la_hvac,229
795,2025-04-06,238,la_hvac,308
796,2025-04-13,306,la_hvac,328
797,2025-04-20,242,la_hvac,301
798,2025-04-27,268,la_hvac,305
799,2025-05-04,249,la_hvac,245


In [111]:
akey['fold'] = 1

In [112]:
akey['horizon'] = list(range(1, 7))

In [113]:
akey

,ds,y,unique_id,baseline,fold,horizon
794,2025-03-30,233,la_hvac,229,1,1
795,2025-04-06,238,la_hvac,308,1,2
796,2025-04-13,306,la_hvac,328,1,3
797,2025-04-20,242,la_hvac,301,1,4
798,2025-04-27,268,la_hvac,305,1,5
799,2025-05-04,249,la_hvac,245,1,6


In [114]:
folds = []
for i in range(12):
    mask = weekly_hvac_permits['ds'] <= origins.iloc[i]
    train = weekly_hvac_permits.loc[mask]
    akey = used_set.iloc[origins.index[i] + 1: origins.index[i] + 7]
    akey_guesses = used_set['y'].iloc[origins.index[i] - (52 - 1) : origins.index[i] - (52 - 7)]
    akey['baseline'] = akey_guesses.tolist()
    akey['fold'] = i + 1
    akey['horizon'] = list(range(1, 7))
    folds.append(akey)

print(len(folds))

12


In [115]:
folds[0]

,ds,y,unique_id,baseline,fold,horizon
794,2025-03-30,233,la_hvac,229,1,1
795,2025-04-06,238,la_hvac,308,1,2
796,2025-04-13,306,la_hvac,328,1,3
797,2025-04-20,242,la_hvac,301,1,4
798,2025-04-27,268,la_hvac,305,1,5
799,2025-05-04,249,la_hvac,245,1,6


In [116]:
used_set['y'].iloc[786]

np.int64(248)

In [117]:
all_folds = pd.concat(folds)

In [118]:
len(all_folds)

72

In [119]:
all_folds.index

Index([794, 795, 796, 797, 798, 799, 798, 799, 800, 801, 802, 803, 802, 803,
       804, 805, 806, 807, 806, 807, 808, 809, 810, 811, 810, 811, 812, 813,
       814, 815, 814, 815, 816, 817, 818, 819, 818, 819, 820, 821, 822, 823,
       822, 823, 824, 825, 826, 827, 826, 827, 828, 829, 830, 831, 830, 831,
       832, 833, 834, 835, 834, 835, 836, 837, 838, 839, 838, 839, 840, 841,
       842, 843],
      dtype='int64')

In [120]:
all_folds

,ds,y,unique_id,baseline,fold,horizon
794,2025-03-30,233,la_hvac,229,1,1
795,2025-04-06,238,la_hvac,308,1,2
796,2025-04-13,306,la_hvac,328,1,3
797,2025-04-20,242,la_hvac,301,1,4
798,2025-04-27,268,la_hvac,305,1,5
...,...,...,...,...,...,...
839,2026-02-08,254,la_hvac,235,12,2
840,2026-02-15,281,la_hvac,261,12,3
841,2026-02-22,281,la_hvac,198,12,4
842,2026-03-01,302,la_hvac,259,12,5


In [121]:
all_folds['miss'] = (all_folds['baseline'] - all_folds['y']).abs()

In [122]:
all_folds.iloc[0]

ds           2025-03-30 00:00:00
y                            233
unique_id                la_hvac
baseline                     229
fold                           1
horizon                        1
miss                           4
Name: 794, dtype: object

In [123]:
all_folds['miss'].iloc[0]

np.int64(4)

In [124]:
MAE_horizon = (all_folds.groupby('horizon')['miss'].sum() / len(folds))

In [125]:
MAE_horizon

horizon
1    22.166667
2    28.666667
3    36.750000
4    35.083333
5    25.416667
6    25.750000
Name: miss, dtype: float64

In [126]:
target_horizon = MAE_horizon * .85

In [127]:
target_horizon

horizon
1    18.841667
2    24.366667
3    31.237500
4    29.820833
5    21.604167
6    21.887500
Name: miss, dtype: float64

In [128]:
mask = weekly_hvac_permits['ds'] <= origins.iloc[0]
train = weekly_hvac_permits.loc[mask]

In [129]:
train

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
789,2025-02-23,198,la_hvac
790,2025-03-02,259,la_hvac
791,2025-03-09,247,la_hvac
792,2025-03-16,258,la_hvac


In [130]:
train_scratch = train['y'].iloc[-52:-46]

In [131]:
train_scratch

742    229
743    308
744    328
745    301
746    305
747    245
Name: y, dtype: int64

In [132]:
def seasonal_naive(train):
  last_year = train['y'].iloc[-52:-46]
  guesses = last_year.to_list()
  return guesses

In [133]:
seasonal_naive(train)

[229, 308, 328, 301, 305, 245]

In [134]:
all_folds['baseline'].iloc[0:6]

794    229
795    308
796    328
797    301
798    305
799    245
Name: baseline, dtype: int64

In [137]:
def score(model):
  folds = []
  for i in range(12):
    mask = weekly_hvac_permits['ds'] <= origins.iloc[i]
    train = weekly_hvac_permits.loc[mask]
    akey = used_set.iloc[origins.index[i] + 1: origins.index[i] + 7]
    akey_guesses = model(train)
    akey['baseline'] = akey_guesses
    akey['fold'] = i + 1
    akey['horizon'] = list(range(1, 7))
    folds.append(akey)
  all_folds = pd.concat(folds)
  all_folds['miss'] = (all_folds['baseline'] - all_folds['y']).abs()
  MAE_horizon = (all_folds.groupby('horizon')['miss'].sum() / len(folds))
  return MAE_horizon

In [138]:
score(seasonal_naive)

horizon
1    22.166667
2    28.666667
3    36.750000
4    35.083333
5    25.416667
6    25.750000
Name: miss, dtype: float64